<a href="https://colab.research.google.com/github/hebanasef99/LBP--face-recognition/blob/main/Lab_workshop_on_LBP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
from pathlib import Path

zip_name = "/content/LBP.zip"  # Change to "/content/lab.zip" if that is its name

with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

%cd /content/LBP

/content/LBP


In [ ]:
!pip -q install numpy scipy pillow

**ORL images**

In [ ]:
import subprocess
import shutil
from pathlib import Path

source = Path("/content/orl_src")

if not source.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/saeid436/Face-Recognition-MLP.git",
        str(source)
    ], check=True)

Path("data").mkdir(exist_ok=True)
shutil.copytree(source / "ORL", "data/ORL", dirs_exist_ok=True)

!python -m lbpface.datasets data/ORL

images      : 400 of size 92x112
subjects    : 40
fingerprint : 9972801133ec8ba8b06221a0f2c7c087c2e2a8d99a5fdd0ddfbf90beb82d5ac3


**Run the workshop baseline**

In [ ]:
instructions = Path("requirements.txt").read_text()
start = instructions.index("import numpy as np")
baseline_code = instructions[start:]

exec(baseline_code)

baseline            0.975
uniform brightness  0.975
lighting ramp 0.2 0.96
lighting ramp 0.4 0.945
lighting ramp 0.6 0.895
lighting ramp 0.8 0.705
rotate 5 deg 0.97
rotate 10 deg 0.925
rotate 20 deg 0.63


In [ ]:
!python -m experiments.orl \
    --data data/ORL \
    --seed 0 \
    --n-perm 100 \
    --out results/orl_reproduction.json

ORL: 400 images, feature length 2187, features+distances 9.1s
mean recognition rate 0.9826  std 0.0096  95% [0.962, 1.000]  (100 permutations)
paper reports         0.9800  std 0.0120
wrote results/orl_reproduction.json   digest 3d349dac87dc50db...


**Improving The Result**

In [ ]:
import numpy as np
import pandas as pd
from scipy import ndimage

from lbpface.datasets import load_orl
from lbpface.features import LBPDescriptor
from lbpface.distances import pairwise_distances
from lbpface.classify import recognition_rate

# Same data, split, and LBP settings as the baseline.
fs = load_orl("data/ORL")
assert fs.images.shape == (400, 112, 92)

desc = LBPDescriptor.orl()

is_gallery = (np.arange(len(fs.images)) % 10) < 5

gal = fs.images[is_gallery]
prb = fs.images[~is_gallery]
gl = fs.subjects[is_gallery]
pl = fs.subjects[~is_gallery]


def illum_ramp(img, strength):
    ramp = 1 + strength * np.linspace(-1, 1, img.shape[1])
    return np.clip(img * ramp, 0, 255)


def rotate(img, degrees):
    return ndimage.rotate(
        img.astype(float),
        degrees,
        reshape=False,
        order=1,
        mode="nearest"
    )


# Original gallery for the baseline comparison.
G_baseline = desc.transform(gal)

# Fixed training augmentation settings.
angles = [-20, -10, 0, 10, 20]
strengths = [-0.8, -0.4, 0, 0.4, 0.8]

feature_batches = []
label_batches = []

for angle in angles:
    rotated_gallery = [
        img if angle == 0 else rotate(img, angle)
        for img in gal
    ]

    for strength in strengths:
        augmented_gallery = [
            img if strength == 0 else illum_ramp(img, strength)
            for img in rotated_gallery
        ]

        feature_batches.append(desc.transform(augmented_gallery))
        label_batches.append(gl.copy())

G_improved = np.concatenate(feature_batches, axis=0)
labels_improved = np.concatenate(label_batches)

print("Original gallery:", len(G_baseline))
print("Improved gallery:", len(G_improved))

Original gallery: 200
Improved gallery: 5000


**Getting result**

In [ ]:
# Exact test conditions from the instructor's code.
conditions = {
    "normal": prb,
    "uniform brightness": [0.5 * img + 20 for img in prb]
}

for strength in [0.2, 0.4, 0.6, 0.8]:
    conditions[f"lighting ramp {strength}"] = [
        illum_ramp(img, strength) for img in prb
    ]

for angle in [5, 10, 20]:
    conditions[f"rotate {angle} deg"] = [
        rotate(img, angle) for img in prb
    ]

rows = []

for condition, test_images in conditions.items():
    probe_features = desc.transform(test_images)

    baseline_accuracy = recognition_rate(
        pairwise_distances(probe_features, G_baseline, "chi2"),
        gl,
        pl
    )

    improved_accuracy = recognition_rate(
        pairwise_distances(probe_features, G_improved, "chi2"),
        labels_improved,
        pl
    )

    rows.append({
        "Condition": condition,
        "Baseline": baseline_accuracy,
        "Improved": improved_accuracy,
        "Gain (percentage points)": (
            improved_accuracy - baseline_accuracy
        ) * 100
    })

    print(
        f"{condition:22s} "
        f"baseline={baseline_accuracy:.3f}  "
        f"improved={improved_accuracy:.3f}"
    )

results = pd.DataFrame(rows)
display(results.round(3))

# Check the professor's goal using your measured results.
scores = results.set_index("Condition")

print(
    "Normal accuracy preserved:",
    scores.loc["normal", "Improved"]
    >= scores.loc["normal", "Baseline"]
)

print(
    "Strong lighting reaches original mild-lighting accuracy:",
    scores.loc["lighting ramp 0.8", "Improved"]
    >= scores.loc["lighting ramp 0.2", "Baseline"]
)

print(
    "20-degree rotation reaches original 5-degree accuracy:",
    scores.loc["rotate 20 deg", "Improved"]
    >= scores.loc["rotate 5 deg", "Baseline"]
)

results.to_csv("lbp_improvement_results.csv", index=False)

normal                 baseline=0.975  improved=0.975
uniform brightness     baseline=0.975  improved=0.975
lighting ramp 0.2      baseline=0.960  improved=0.975
lighting ramp 0.4      baseline=0.945  improved=0.975
lighting ramp 0.6      baseline=0.895  improved=0.970
lighting ramp 0.8      baseline=0.705  improved=0.970
rotate 5 deg           baseline=0.970  improved=0.985
rotate 10 deg          baseline=0.925  improved=0.990
rotate 20 deg          baseline=0.630  improved=0.980


,Condition,Baseline,Improved,Gain (percentage points)
0,normal,0.975,0.975,0.0
1,uniform brightness,0.975,0.975,0.0
2,lighting ramp 0.2,0.960,0.975,1.5
3,lighting ramp 0.4,0.945,0.975,3.0
4,lighting ramp 0.6,0.895,0.970,7.5
5,lighting ramp 0.8,0.705,0.970,26.5
6,rotate 5 deg,0.970,0.985,1.5
7,rotate 10 deg,0.925,0.990,6.5
8,rotate 20 deg,0.630,0.980,35.0


Normal accuracy preserved: True
Strong lighting reaches original mild-lighting accuracy: True
20-degree rotation reaches original 5-degree accuracy: True
